# Linear and nonlinear matter power spectrum from CLASS $+$ halofit

Upstream data-generation notebook for `matter_power.ipynb`.

Runs **CLASS** for the linear and `halofit` nonlinear matter power spectrum out to
$k = 100\,\mathrm{Mpc^{-1}}$, then extrapolates both to $k = 10^{10}\,h/\mathrm{Mpc}$
with a self-contained re-implementation of the Takahashi+2012 `halofit`
prescription (cross-checked against the CLASS output over the shared range).

**Outputs** (both committed to the repository, so `matter_power.ipynb` runs
without CLASS):

- `Delta2_output.csv` --- `k [1/Mpc], k [h/Mpc], Delta2_lin, Delta2_nl` over the
  CLASS range;
- `Delta2_extended_output.csv` --- the same columns on the extended grid, plus
  `Delta2_nl_30`, the nonlinear spectrum truncated at $k = 30\,h/\mathrm{Mpc}$
  (the thick black curve of `figs/matter_power.pdf`).

**Requires `classy`**, the CLASS Python wrapper, which is *not* part of the
`environment.yml` environment (it needs a compiled CLASS). Install it only if you
want to regenerate the CSVs:

```bash
pip install classy    # or build CLASS from source and `make classy`
```

*Runtime: a few minutes.*

# Setup

## Packages

In [ ]:
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt
from classy import Class
from time import perf_counter
import os
from natural_units_GeV import *

os.environ.setdefault("OMP_NUM_THREADS", "1")

plt.rcdefaults()
fontsize = 12
from matplotlib import font_manager
from matplotlib import rcParams
from matplotlib import rc
rcParams['font.family'] = 'serif'
font_manager.findfont('serif', rebuild_if_missing=True)
rcParams.update({'font.size':fontsize})
rc('text', usetex=True)
custom_preamble = {
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}"
    }
plt.rcParams.update(custom_preamble)

## Controls

In [ ]:
z = 0.0                   # redshift
kmin = 1e-4               # CLASS units: 1/Mpc
kmax = 1e2                # max k for CLASS (1/Mpc)
num_k = 2000              # sampling density

# optional: specify cosmology explicitly (uncomment & edit)
# base_cosmo = {
#     'h': 0.674, 'Omega_b': 0.049, 'Omega_cdm': 0.265,
#     'A_s': 2.1e-9, 'n_s': 0.965, 'tau_reio': 0.054
# }
base_cosmo = {}

# precision knobs for smoother high-k curves
precision = {
    'P_k_max_1/Mpc': kmax,
    'k_per_decade_for_pk': 50,   # densify CLASS grid for P(k)
    # 'k_bao_cut': 1e4
    # add more if you push extremely high k:
    # 'k_step_sub': 0.01,
    # 'k_per_decade_for_bao': 60,
}

extra = {
    'perturbations_verbose': 2,  # 0: no output, 1: some output, 2: all output
    'fourier_verbose': 2,  # 0: no output, 1: some output, 2: all output
    'output': 'mPk',  # output only the matter power spectrum
    'non linear': "Halofit",  # Halofit or HMcode
}

In [ ]:
cosmo_nl = Class()
cosmo_nl.set({**base_cosmo, **precision, **extra})
cosmo_nl.compute()

In [ ]:
Omega_m0 = cosmo_nl.get_current_derived_parameters(['Omega_m'])['Omega_m']
h = cosmo_nl.h()
rho_crit = 3 * m_Planck**2 * (h * 100 * km / second / Mpc)**2 
rho_m = Omega_m0 * rho_crit
print("Omega_m0 = {:.4g}".format(Omega_m0))   
print("h = {:.4g}".format(h))
print("rho_crit = {:.4g} M_Solar pc^-3".format(rho_crit / (M_Solar / pc**3)))
print("rho_m = {:.4g} M_Solar pc^-3".format(rho_m / (M_Solar / pc**3)))

## Helpers

In [ ]:
def to_h_units(k_1_per_Mpc, P_Mpc3, h):
    """Return k [h/Mpc], P [(Mpc/h)^3]."""
    return k_1_per_Mpc / h, P_Mpc3 * h**3

def delta2(k_hMpc, P_h3):
    """Dimensionless power Delta^2_delta(k) = k^3 P(k)/(2 pi^2), k in h/Mpc, P in (Mpc/h)^3."""
    return (k_hMpc**3) * P_h3 / (2.0 * np.pi**2)

def dlnD2_dlnk(k, D2_lin, clip=1e-300):
    """
    Return d ln(Δ^2_lin)/d ln k on the same grid.
    Works on arbitrary (strictly increasing) k; if ln k is uniform,
    it uses a constant spacing for slightly lower noise.
    """
    k = np.asarray(k, float)
    D2 = np.asarray(D2_lin, float)
    if k.ndim != 1 or D2.shape != k.shape or np.any(k <= 0):
        raise ValueError("k and D2_lin must be 1D, same shape, with k>0.")
    x = np.log(k)
    y = np.log(np.maximum(D2, clip))

    dx = np.diff(x)
    if np.allclose(dx, dx[0], rtol=1e-6, atol=0):   # uniform in ln k (e.g., geomspace)
        return np.gradient(y, dx[0], edge_order=2)
    else:                                           # non-uniform ln k
        return np.gradient(y, x, edge_order=2)

# Computation

## CLASS

In [ ]:
kk = np.geomspace(kmin, kmax, num_k)  # CLASS units (1/Mpc)

# Non-linear spectrum
t0 = perf_counter()               # <-- start timing
cosmo_nl = Class()
cosmo_nl.set({**base_cosmo, **precision, **extra})
cosmo_nl.compute()
t1 = perf_counter()               # <-- end timing
print(f"Cosmo compute took {t1 - t0:.2f} s\n")
Pk_lin = np.array([cosmo_nl.pk_lin(k, z) for k in kk])  # Mpc^3
Pk_nl = np.array([cosmo_nl.pk(k, z) for k in kk])        # Mpc^3
t2 = perf_counter()               # <-- end timing
print(f"Pk compute took {t2 - t1:.2f} s")

# Units & Δ²
h = cosmo_nl.h()
k_h, P_lin_h3 = to_h_units(kk, Pk_lin, h)
_,   P_nl_h3  = to_h_units(kk, Pk_nl,  h)
D2_lin = delta2(k_h, P_lin_h3)
D2_nl  = delta2(k_h, P_nl_h3)

In [ ]:
data = np.column_stack([kk, k_h, D2_lin, D2_nl])
header = "k[1/Mpc], k[h/Mpc], Delta2_lin, Delta2_nl"
np.savetxt("Delta2_output.csv", data, delimiter=",", header=header, comments='')

## Extrapolate

In [ ]:
data = np.loadtxt("Delta2_output.csv", delimiter=",", skiprows=1)  # skip header
kk, k_h, D2_lin, D2_nl = data.T

In [ ]:
h = cosmo_nl.h()
print('h = {:.3f}'.format(h))
kmax_ext = 1e10 # kmax for extrapolation
N_ext = 1000 # number of points for extrapolation
kk_ext = np.concatenate((kk,np.geomspace(kk[-1], kmax_ext, N_ext+1)[1:])) # add 1000 points to end
k_h_ext = kk_ext / h # convert to h units

# Extend the linear spectrum by holding Delta^2_lin flat above the CLASS range. This is the
# nearly scale-invariant small-scale plateau of Eq. (growth) (Sec. IV A): the extra
# radiation-era growth of smaller modes nearly cancels the red primordial tilt.
D2_lin_ext = np.concatenate((D2_lin, D2_lin[-1]*np.ones(N_ext)))

In [ ]:
# halofit truncated at k = 30 h/Mpc -- the thick black curve of figs/matter_power.pdf, beyond
# which the one-halo model of matter_power.ipynb takes over
D2_nl_30 = np.zeros_like(k_h_ext)
D2_nl_30[k_h_ext < 30] = D2_nl[k_h < 30]
D2_nl_30[k_h_ext >= 30] = D2_nl[k_h < 30][-1]

In [ ]:
slope = dlnD2_dlnk(k_h, D2_lin)   # this is d ln Δ^2_lin / d ln k

fig,ax = plt.subplots(figsize=(6, 4))
ax.plot(k_h, slope, label=r"$d \ln \Delta^2_{\rm lin}/d \ln k$", color='k')
ax.set_xlabel(r"$k \, [h/{\rm Mpc}]$")
ax.set_ylabel(r"$d \ln \Delta^2_{\rm lin}/d \ln k$")
ax.set_xscale("log")
ax.set_yscale("linear")
ax.set_ylim(-1,4)
ax.set_xlim(1e-4,1e5)
ax.grid()

In [ ]:
@dataclass
class CosmologyFlat:
    """Flat wCDM background needed by HALOFIT-T12 for f1,f2,f3 and w-terms."""
    Omega_m0 = cosmo_nl.get_current_derived_parameters(['Omega_m'])['Omega_m']
    w: float = -1.0              # constant dark-energy equation of state

    def Ez2(self, z):
        Om0, w = self.Omega_m0, self.w
        Ow0 = 1.0 - Om0
        return Om0*(1+z)**3 + Ow0*(1+z)**(3*(1+w))

    def Omega_m_z(self, z):
        Om0 = self.Omega_m0
        return Om0*(1+z)**3 / self.Ez2(z)

    def Omega_w_z(self, z):
        return 1.0 - self.Omega_m_z(z)

In [ ]:
def halofit_t12(k, delta2_lin, z, cosmo=CosmologyFlat()):
    """
    Takahashi et al. (2012) HALOFIT mapping: Δ^2_lin(k) -> Δ^2_nl(k).
    Inputs
    ------
    k : array_like
        Wavenumbers (units arbitrary but consistent across inputs).
    delta2_lin : array_like
        Linear dimensionless power Δ^2_lin(k) on the same k grid.
    z : float
        Redshift.
    cosmo : CosmologyFlat
        Flat wCDM background (Ω_m0, w). Ω_k=0 is assumed.

    Returns
    -------
    delta2_nl : ndarray
        Nonlinear dimensionless power Δ^2_nl(k) on the same grid.
    """
    k = np.asarray(k, dtype=float)
    Dl = np.asarray(delta2_lin, dtype=float)
    if np.any(k <= 0) or k.ndim != 1:
        raise ValueError("k must be a 1D array of positive values.")
    if Dl.shape != k.shape:
        raise ValueError("delta2_lin must have the same shape as k.")

    # --- helpers on a log-k grid ---
    lnk = np.log(k)

    # sigma^2(R) with the Gaussian window of T12: int dln k Delta^2_lin(k) exp[-(kR)^2]
    # (NB Gaussian, not the real-space top-hat used for the halo mass function elsewhere)
    def sigma2_of_R(R):
        """Calculate sigma^2(R) using the linear power spectrum."""
        integrand = Dl * np.exp(-(k*R)**2)
        return np.trapezoid(integrand, lnk)
    
    # Find R_sigma such that σ^2(R_sigma)=1 (robust bracket+bisect in ln R)
    # Bracket across many decades around the physical range set by k
    Rmin = 1.0 / (k.max())
    Rmax = 1.0 / (k.min())
    lnR_lo, lnR_hi = np.log(Rmin), np.log(Rmax)
    # Ensure the bracket actually straddles 1
    s_lo = sigma2_of_R(np.exp(lnR_lo))
    s_hi = sigma2_of_R(np.exp(lnR_hi))
    # If not straddling, expand the bracket
    expand = 0
    while not ((s_lo-1.0)*(s_hi-1.0) < 0.0) and expand < 20:
        lnR_lo -= 1.0
        lnR_hi += 1.0
        s_lo = sigma2_of_R(np.exp(lnR_lo))
        s_hi = sigma2_of_R(np.exp(lnR_hi))
        expand += 1
    if (s_lo-1.0)*(s_hi-1.0) >= 0.0:
        # fallback: monotonic guess (common for very steep spectra)
        lnR_sigma = 0.5*(lnR_lo + lnR_hi)
    else:
        for _ in range(100):
            mid = 0.5*(lnR_lo + lnR_hi)
            s_mid = sigma2_of_R(np.exp(mid))
            if (s_lo-1.0)*(s_mid-1.0) < 0.0:
                lnR_hi = mid
                s_hi = s_mid
            else:
                lnR_lo = mid
                s_lo = s_mid
        lnR_sigma = 0.5*(lnR_lo + lnR_hi)
    R_sigma = np.exp(lnR_sigma)
    k_sigma = 1.0 / R_sigma
    y = k / k_sigma
    # print('k_sigma = ', str(k_sigma)[0:6])

    # Effective slope n_eff and curvature C using finite differences in ln R
    h = 1e-3
    s_m = sigma2_of_R(np.exp(lnR_sigma - h))
    s_0 = sigma2_of_R(np.exp(lnR_sigma))
    s_p = sigma2_of_R(np.exp(lnR_sigma + h))
    dlns_dlnR = (np.log(s_p) - np.log(s_m)) / (2*h)
    d2lns_dlnR2 = (np.log(s_p) - 2*np.log(s_0) + np.log(s_m)) / (h*h)

    n_eff = -3.0 - dlns_dlnR
    C = - d2lns_dlnR2

    # --- Coefficients (T12 Appendix A; equations A6–A13) ---
    Om = cosmo.Omega_m_z(z)
    Ow = cosmo.Omega_w_z(z)
    w = cosmo.w

    log10a = (1.5222 + 2.8553*n_eff + 2.3706*n_eff**2 + 0.9903*n_eff**3
              + 0.2250*n_eff**4 - 0.6038*C + 0.1749*Ow*(1.0 + w))
    log10b = (-0.5642 + 0.5864*n_eff + 0.5716*n_eff**2 - 1.5474*C
              + 0.2279*Ow*(1.0 + w))
    log10c = (0.3698 + 2.0404*n_eff + 0.8161*n_eff**2 + 0.5869*C)
    gamma_n = (0.1971 - 0.0843*n_eff + 0.8460*C)
    alpha_n = np.abs(6.0835 + 1.3373*n_eff - 0.1959*n_eff**2 - 5.5274*C)
    beta_n  = (2.0379 - 0.7354*n_eff + 0.3157*n_eff**2 + 1.2490*n_eff**3
               + 0.3980*n_eff**4 - 0.1682*C)
    mu_n = 0.0   # T12 sets log10(mu_n) -> -inf, i.e. mu_n = 0 (unlike Smith+ 2003)
    log10nu = (5.2105 + 3.6902*n_eff)

    a_n = 10.0**log10a
    b_n = 10.0**log10b
    c_n = 10.0**log10c
    nu_n = 10.0**log10nu

    # Small functions of Ω_m(z) (Eq. A14)
    f1 = Om**(-0.0307)
    f2 = Om**(-0.0585)
    f3 = Om**( 0.0743)

    # --- Two-halo (quasi-linear) term Δ_Q^2 (Eq. A2) ---
    def f_damp(y):
        return y/4 + y**2/8

    Dq = Dl * ((1.0 + Dl)**beta_n / (1.0 + alpha_n*Dl)) * np.exp(-f_damp(y))

    # --- One-halo term Δ_H^2 (Eq. A3) ---
    Dh_prime = (a_n * y**(3.0*f1)) / (1.0 + b_n*y**f2 + (c_n*f3*y)**(3.0 - gamma_n))
    Dh = Dh_prime / (1.0 + mu_n/y + nu_n/(y*y))

    delta2_nl = Dq + Dh
    return delta2_nl

# ---- convenience wrappers ----
def halofit_t12_from_Plin(k, P_lin, z, cosmo=CosmologyFlat()):
    """If you have P_lin(k) instead, convert to Δ^2 and back."""
    k = np.asarray(k, float)
    delta2_lin = (k**3 * np.asarray(P_lin, float)) / (2.0*np.pi**2)
    return halofit_t12(k, delta2_lin, z, cosmo)

def delta2_to_P(delta2, k):
    """Convert Δ^2(k) back to P(k)."""
    k = np.asarray(k, float)
    return (2.0*np.pi**2 / k**3) * np.asarray(delta2, float)

In [ ]:
D2_nl_manual = halofit_t12(kk, D2_lin, 0.0)         # manual extrapolation as check
D2_nl_ext = halofit_t12(kk_ext, D2_lin_ext, 0.0)    # extrapolated

# Plot

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(12,6))

xlim = (1e-4,1e10); ax.set_xlim(xlim);
ylim = (1e-2,1e12); ax.set_ylim(ylim);
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'wavenumber $k~[h/\mathrm{Mpc}]$');
ax.set_ylabel(r'matter power spectrum $\Delta^2(k) = k^3 P(k) / (2 \pi^2)$');
ax.set_xticks(np.logspace(np.log10(xlim[0]),np.log10(xlim[1]),1+int(np.log10(xlim[1]/xlim[0]))),minor=False);
ax.set_xticks(np.outer(np.logspace(np.log10(xlim[0]),np.log10(xlim[1])-1,int(np.log10(xlim[1]/xlim[0]))),range(1,10)).flatten(),minor=True);
ax.set_yticks(np.logspace(np.log10(ylim[0]),np.log10(ylim[1]),1+int(np.log10(ylim[1]/ylim[0]))),minor=False);
ax.set_yticks(np.outer(np.logspace(np.log10(ylim[0]),np.log10(ylim[1])-1,int(np.log10(ylim[1]/ylim[0]))),range(1,10)).flatten(),minor=True);

ax.plot(k_h,        D2_lin,     color='k', lw = 1, ls='solid', label=r'linear $\Lambda$CDM')
ax.plot(k_h_ext,    D2_lin_ext, color='k', lw = 1, ls='dashed')

# ax.plot(k_h, D2_nl,     color='k', lw = 2, ls='solid', label=r'non-linear $\Lambda$CDM')
# ax.plot(k_h, D2_nl_manual, color='red', lw = 3, ls='dotted', label=r'non-linear $\Lambda$CDM (extrapolated)')
ax.plot(k_h_ext[k_h_ext < 30],  D2_nl_ext[k_h_ext < 30],     color='k', lw = 2, ls='solid', label=r'non-linear $\Lambda$CDM')
ax.plot(k_h_ext[k_h_ext >= 30], D2_nl_ext[k_h_ext >= 30],      color='k', lw = 2, ls='dashed')
ax.plot(k_h_ext[k_h_ext >= 30], D2_nl_30[k_h_ext >= 30],      color='k', lw = 2, ls='dotted')

ax.legend(loc='upper left', frameon=False);
ax.grid();


# Output

In [ ]:
# save extended data
data_ext = np.column_stack([kk_ext, k_h_ext, D2_lin_ext, D2_nl_ext, D2_nl_30])
header_ext = "k[1/Mpc], k[h/Mpc], Delta2_lin, Delta2_nl, Delta2_nl_30"
np.savetxt("Delta2_extended_output.csv", data_ext, delimiter=",", header=header_ext, comments='')